In [ ]:
## View ESP of molecular moment
fchk_file='run_psi4/SP_PBE0_aug-cc-pvtz_h2o.fchk'
mol_mom='run_psi4/molmom.mom'
dis_mom='horton_run/moment.mom'

import os
from qcp_MolView.main import plot_esp_surface
from python_scripts.run_process import run_subrocess

from python_scripts.conda_env import get_python_for_script
from qcp_global_utils.environment.file_handling import pdtc_file
from pydantic import validate_call; val_call=validate_call(config=dict(arbitrary_types_allowed=True, validate_return=True))

from contextlib import contextmanager
import time

ModuleNotFoundError: No module named 'mayavi_molecule_viewer'

In [ ]:

@contextmanager
def run_info(tag):
    start=time.time()
    print(f"#-- Starting {tag}")
    try:
        yield
    finally:
        print(f"--- Finished after {time.time()-start:.2f} [s]")


@val_call
def make_surface(fchk_file:pdtc_file, grid_spacing:float=0.25, iso_density: float=0.001) -> pdtc_file:
    """ Returns file name of surface 
    
    """
    python,script=get_python_for_script('qcp_dens','calc_IsoDensSurf.py')

    import subprocess as sp
    cmd=f"{python} {script} {os.path.realpath(fchk_file)} --grid_spacing {grid_spacing} --iso_dens {iso_density}"
    run_subrocess(cmd, 'make_surface')
    return os.path.realpath('make_surface/dummy_surface.json')

@val_call
def make_densesp_map(fchk_file, surface_file) -> pdtc_file:
    python,script=get_python_for_script('qcp_dens','calc_esp.py')
    run_dir,ofile="make_dens_esp","dens_esp.json"
    cmd=f"{python} {script} --dens {os.path.realpath(fchk_file)} --surf {os.path.realpath(surface_file)} --engine numpy -o {ofile}"
    run_subrocess(cmd, run_dir)
    return os.path.realpath(os.path.join(run_dir,ofile))

@val_call
def make_mul_map(moment_file:pdtc_file, surface_file:pdtc_file=None, run_dir='make_mom_esp') -> pdtc_file:
    """ Could play with rank """
    python,script=get_python_for_script('qcp_dens','calc_mul_esp.py')
    ofile='mom_esp.json'
    cmd=f"{python} {script} --mom {os.path.realpath(moment_file)} --grid {os.path.realpath(surface_file)}  --ranks \'max4\' -o {ofile}"
    run_subrocess(cmd, run_dir)
    return os.path.realpath(os.path.join(run_dir,ofile))


# make context window with info and timing
with run_info('run_surface'):
    surface_file=make_surface(fchk_file)
with run_info('run_surface'):
    esp_map=make_densesp_map(fchk_file, surface_file)
with run_info('run_surface'):
    molmom_map=make_mul_map(mol_mom,surface_file, run_dir='make_molmom_esp')
with run_info('run_surface'):
    dismom_map=make_mul_map(dis_mom,surface_file, run_dir='make_dismom_esp')

#-- Starting run_surface
Started process with PID 11614
--- Finished after 1.68 [s]
#-- Starting run_surface
Started process with PID 11637
--- Finished after 18.85 [s]
#-- Starting run_surface
Started process with PID 11701
--- Finished after 0.32 [s]
#-- Starting run_surface
Started process with PID 11704
--- Finished after 0.32 [s]


In [ ]:
# So first 
from mayavi import mlab
import os
mlab.init_notebook(backend='ipy') # Allows for interactive matlab plots
print(dismom_map, esp_map, fchk_file)
view_object=plot_esp_surface(esp=molmom_map,esp_ref=esp_map, geom=fchk_file,
                            #opacity=0.75, see_through=one_side_see_through
)
view_object[0]

Notebook initialized with ipy backend.
/home/bruno/1_PhD/6_Other/06-03_Collaborations/run_horton_for_workshop/make_dismom_esp/mom_esp.json /home/bruno/1_PhD/6_Other/06-03_Collaborations/run_horton_for_workshop/make_dens_esp/dens_esp.json run_psi4/SP_PBE0_aug-cc-pvtz_h2o.fchk


Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01\xf4\x00\x00\x01\xf4\x08\x02\x00\x00\x00D\xb4H\xd…